In [3]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from utils import simulate_bagging_and_ijk_var_calculation

In [4]:
####### Simulation parameters  #########
n_x = 1_000 
n_sim = 1_000
B = 10_000
seed = 42
portion_non_zero_weights = 1.
var_x = 1
m = int(n_x*portion_non_zero_weights)
folder_name = "weighted_bagged_mean_estimator"

# Data for Simulation
rng = np.random.default_rng(seed)
x_sim = rng.normal(0,var_x**0.5,(n_sim, n_x))
weights = np.zeros(n_x)
weights[:m] = 1/m

true_var = n_x/m**2 * var_x

In [ ]:
theta_bagged= np.zeros(n_sim)
theta_bagged_var_ijk = np.zeros(n_sim)

# run simulations
with ProcessPoolExecutor() as executor:
    futures = [
        executor.submit(
            simulate_bagging_and_ijk_var_calculation,
            x1=x_sim[i],
            B=B,
            sim_i=i,
            seed=seed,
            weights=weights,
            m=m
        )
        for i in range(n_sim)
    ]

    for i, future in enumerate(tqdm(futures, desc="Simulations", unit="simulation")):
        _theta_bagged_var_ijk, _theta_bagged = future.result()
        theta_bagged[i] =_theta_bagged
        theta_bagged_var_ijk[i] = _theta_bagged_var_ijk
        
        

In [6]:
print(f"\nMean of UNbiased_ijk_var_est of bagged_mean_estimator: {round(theta_bagged_var_ijk.mean(),5)}")
print(f"True variance of bagged_mean_estimator: {true_var}\n")

print(f"Mean of bagged_mean_estimator: {round(theta_bagged.mean(),5)}")
print(f"True mean of bagged_mean_estimator: {0}\n")


Mean of UNbiased_ijk_var_est of bagged_mean_estimator: 0.001
True variance of bagged_mean_estimator: 0.001

Mean of bagged_mean_estimator: 0.0001
True mean of bagged_mean_estimator: 0

